In [1]:
!rm -f minsearch.py
!wget https://raw.githubusercontent.com/alexeygrigorev/minsearch/main/minsearch.py

--2025-01-07 08:10:39--  https://raw.githubusercontent.com/alexeygrigorev/minsearch/main/minsearch.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
connected. to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... 
HTTP request sent, awaiting response... 200 OK
Length: 3832 (3.7K) [text/plain]
Saving to: ‘minsearch.py’

minsearch.py        100%[===================>]   3.74K  --.-KB/s    in 0s      

2025-01-07 08:10:39 (21.2 MB/s) - ‘minsearch.py’ saved [3832/3832]



In [1]:
import requests 
import minsearch

docs_url = 'https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-intro/documents.json?raw=1'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

index = minsearch.Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

index.fit(documents)

In [2]:
def search(query):
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5
    )
    return results

In [3]:
def build_prompt(query, search_results):
    prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT: 
{context}
""".strip()

    context = ""
    
    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
    
    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt

def llm(prompt):
    response = client.chat.completions.create(
        model='phi3',
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [4]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [5]:
from openai import OpenAI

client = OpenAI(
    base_url='http://localhost:11434/v1/',
    api_key='ollama',
)

In [6]:
llm('write that this is a test')

'This document serves as an official record to verify the current status. It tests whether communication systems are operational and if they meet quality standards required for reliable performance. This examination ensures compliance with established guidelines. (Note: Since there\'inis no real "system" or specifics given, this is a generic placeholder text.)\n\nPlease note that due to the lack of context in your request about what kind of system we are testing and which quality standards need to be met, I assumed it was an operational communication test. If you have specific systems, metrics, or guidelines relating to the tests required for this document\'s purpose, they should replace the placeholders herein.\n\n--- \n\nIn a real-world application where such system testing might be necessary within your company using Java programming language with TestNG framework and Maven project structure:\n\n```java\nimport org.testng.annotations.*;\nimport static org.junit.Assert.*;\n\nclass Co

In [7]:
print(_)

This document serves as an official record to verify the current status. It tests whether communication systems are operational and if they meet quality standards required for reliable performance. This examination ensures compliance with established guidelines. (Note: Since there'inis no real "system" or specifics given, this is a generic placeholder text.)

Please note that due to the lack of context in your request about what kind of system we are testing and which quality standards need to be met, I assumed it was an operational communication test. If you have specific systems, metrics, or guidelines relating to the tests required for this document's purpose, they should replace the placeholders herein.

--- 

In a real-world application where such system testing might be necessary within your company using Java programming language with TestNG framework and Maven project structure:

```java
import org.testng.annotations.*;
import static org.junit.Assert.*;

class CommunicationTest